# PLZ → NUTS3 Linkage Pipeline

**Purpose:** This notebook demonstrates the end-to-end process of merging German postal code (PLZ) data from FHIR-structured clinical records with NUTS3 regional boundaries, computing calendar-week timestamps, and deriving patient ages at key clinical events.

It serves as the reference implementation for the production script [`link_plz_nuts3.py`](link_plz_nuts3.py). In addion, the script reads and merges three defined FHIR CSV files and extracts the a dataset of interest as merged DataFrame(doi). Each section corresponds directly to a step in that script.

---

**Input files expected** (relative to the notebook working directory):
- `test_data_filled/Diagnose.csv` — FHIR Condition resources
- `test_data_filled/KontaktGesundheitseinrichtung.csv` — FHIR Encounter resources
- `test_data_filled/PatientPseudonymisiert.csv` — FHIR Patient pseudonymised resources
- `PLZ_Gebiete.gpkg` — GeoPackage of 5-digit PLZ polygons with a plz column
- `nuts250_12-31.gk3.shape.zip` — ZIP archive containing the NUTS3 boundary shapefile from BKG

**Output:** A merged DataFrame (`dOI`) with one row per diagnosis encounter, containing regional assignment (NUTS3), calendar-week timestamps, and derived ages — ready for epidemiological analysis.

## 1. Imports

Core libraries used throughout this notebook:
- **pandas** — tabular data manipulation
- **geopandas** — spatial joins and GeoDataFrame operations
- **zipfile** — extracting the NUTS3 shapefile from its ZIP archive
- **numpy** — numerical operations (floor division for age calculation)

In [1]:
import pandas as pd
import geopandas as gpd
import zipfile
import numpy as np

## 2. Helper: Date Parsing

FHIR date fields are inconsistently formatted across resources — some records carry full ISO-8601 timestamps with timezone offsets, others only a year or year-month. The helper `parse_datum` tries each format in order from most specific to least specific and returns a timezone-naive, time-stripped `datetime` (i.e. midnight on the given date).

Missing or blank values are returned as `pd.NaT` so downstream calculations propagate nulls cleanly.

In [2]:
def parse_datum(val):
    if pd.isna(val) or str(val).strip() == "":
        return pd.NaT
    val = str(val).strip()
    for fmt in [
        "%Y-%m-%dT%H:%M:%S%z",  # 2023-01-15T10:30:00+02:00
        "%Y-%m-%d",              # 2023-01-15
        "%Y-%m",                 # 2023-01
        "%Y",                    # 2023
    ]:
        try:
            dt = pd.to_datetime(val, format=fmt)
            # Always return date only (no time, no timezone)
            return dt.normalize().tz_localize(None) if dt.tzinfo is None else dt.tz_convert(None).normalize()
        except ValueError:
            continue
    return pd.NaT

## 3. Load and Prepare Clinical Data

Three FHIR-derived CSV files are read and normalised before merging:

| Source file | Key columns selected | Notes |
|---|---|---|
| `PatientPseudonymisiert.csv` | `id`, `Patient_gender`, `Patient_birthDate`, `Patient_addressStrassenanschrift_postalCode` | PLZ read as string to preserve leading zeros |
| `Diagnose.csv` | `id`, `patient`, `Condition_encounter_reference`, `Condition_recordedDate`, `Condition_extensionFeststellungsdatum_value_X_Valuedatetime` | Reference prefixes (`Patient/`, `Encounter/`) stripped |
| `KontaktGesundheitseinrichtung.csv` | `id`, `patient`, `Encounter_diagnosis_condition_reference`, `Encounter_period_start` | Reference prefixes stripped |

Some of the columns are renamed as follows:
| Source file | original column name | new column name |
|---|---|---|
| `PatientPseudonymisiert.csv` | `id` | `patient_id` |
| `PatientPseudonymisiert.csv` | `Patient_addressStrassenanschrift_postalCode` | `plz` |
| `Diagnose.csv` | `id` | `condition_id` |
| `Diagnose.csv` | `patient` | `patient_id` |
| `Diagnose.csv` | `Condition_encounter_reference` | `encounter_id` |
| `Diagnose.csv` | `Condition_extensionFeststellungsdatum_value_X_Valuedatetime` | `Feststellungsdatum` |
| `KontaktGesundheitseinrichtung.csv` | `id` | `encounter_id` |
| `KontaktGesundheitseinrichtung.csv` | `patient` | `patient_id` |
| `KontaktGesundheitseinrichtung.csv` | `Encounter_diagnosis_condition_reference` | `condition_id` |
```
The three tables are joined as:
```
Diagnose ──(patient_id)──▶ PatientPseudonymisiert
         ──(encounter_id)──▶ KontaktGesundheitseinrichtung  [left join]
```

A numeric surrogate `id` is assigned per patient (via `pd.factorize`) so the output contains no original identifiers. All date columns are parsed with `parse_datum` to ensure a uniform, timezone-naive datetime type.

> **Note:** The actual input files with real disease occurrences should contain **one entry per patient** listing data refering to the **earliest diagnosis date**.

In [3]:
diagnose = pd.read_csv("test_data_filled/Diagnose.csv")
kontaktGesundheitseinrichtung = pd.read_csv("test_data_filled/KontaktGesundheitseinrichtung.csv")
patientPseudonymisiert = pd.read_csv("test_data_filled/PatientPseudonymisiert.csv",dtype={"Patient_addressStrassenanschrift_postalCode": str})

# 1st patientPseudonymisiert
patientPseudonymisiert = patientPseudonymisiert[['id', 'Patient_gender','Patient_birthDate','Patient_addressStrassenanschrift_postalCode']]
patientPseudonymisiert.rename(columns={'Patient_addressStrassenanschrift_postalCode':'plz'}, inplace=True)
patientPseudonymisiert.rename(columns={'id':'patient_id'}, inplace=True)

# 2nd diagnose
diagnose['patient'] = diagnose['patient'].str.replace('Patient/', '')
diagnose['Condition_encounter_reference'] = diagnose['Condition_encounter_reference'].str.replace('Encounter/', '')
diagnose = diagnose[['id','patient','Condition_recordedDate','Condition_encounter_reference','Condition_extensionFeststellungsdatum_value_X_Valuedatetime']]
diagnose.rename(columns={'patient':'patient_id'}, inplace=True)
diagnose.rename(columns={'id':'condition_id'}, inplace=True)
diagnose.rename(columns={'Condition_encounter_reference':'encounter_id'}, inplace=True)
diagnose.rename(columns={'Condition_extensionFeststellungsdatum_value_X_Valuedatetime':'Feststellungsdatum'}, inplace=True)

# 3rd kontaktGesundheitseinrichtung
kontaktGesundheitseinrichtung['patient'] = kontaktGesundheitseinrichtung['patient'].str.replace('Patient/', '')
kontaktGesundheitseinrichtung['Encounter_diagnosis_condition_reference'] = kontaktGesundheitseinrichtung['Encounter_diagnosis_condition_reference'].str.replace('Condition/', '')
kontaktGesundheitseinrichtung = kontaktGesundheitseinrichtung[['id','patient','Encounter_period_start','Encounter_diagnosis_condition_reference']]
kontaktGesundheitseinrichtung.rename(columns={'patient':'patient_id'}, inplace=True)
kontaktGesundheitseinrichtung.rename(columns={'id':'encounter_id'}, inplace=True)
kontaktGesundheitseinrichtung.rename(columns={'Encounter_diagnosis_condition_reference':'condition_id'}, inplace=True)

dOI = pd.merge( pd.merge(diagnose, patientPseudonymisiert, on='patient_id'), kontaktGesundheitseinrichtung, how='left', on='encounter_id')
dOI["id"] = pd.factorize(dOI["patient_id_x"])[0]
dOI = dOI[['id','plz','Encounter_period_start','Condition_recordedDate','Patient_gender','Patient_birthDate','Feststellungsdatum']]

dOI["Patient_birthDate"] = dOI["Patient_birthDate"].apply(parse_datum)
dOI["Encounter_period_start"] = dOI["Encounter_period_start"].apply(parse_datum)
dOI["Condition_recordedDate"] = dOI["Condition_recordedDate"].apply(parse_datum)
dOI["Feststellungsdatum"] = dOI["Feststellungsdatum"].apply(parse_datum)

dOI

,id,plz,Encounter_period_start,Condition_recordedDate,Patient_gender,Patient_birthDate,Feststellungsdatum
0,0,1011,NaT,2024-08-15,male,1969-01-01,2019-05-14
1,0,1011,2019-05-14,2019-05-14,male,1969-01-01,2019-05-14
2,1,20095,2023-10-12,2023-10-12,male,1939-01-01,2019-05-14
3,2,80331,2022-11-08,2022-11-08,male,1994-01-01,2019-05-14
4,3,40210,2024-06-02,2024-06-02,other,1981-01-01,2019-05-14
5,4,50667,2021-09-12,2021-09-12,female,1982-01-01,2019-05-14
6,5,6031,2017-06-19,2017-06-19,male,1977-01-01,2019-05-01
7,6,70173,2022-10-05,2022-10-05,female,1958-01-01,2019-05-01
8,7,30159,2024-02-20,2024-02-20,female,1981-01-01,NaT
9,8,010,2023-04-09,2023-04-09,other,1970-01-01,NaT


## 4. Load Geospatial Reference Data

Two geospatial layers are loaded:

1. **PLZ polygons** (`PLZ_Gebiete.gpkg`): GeoPackage containing 5-digit postal code areas for Germany. Each feature has a `plz` attribute.
2. **NUTS3 polygons** (`NUTS250_N3.shp`): NUTS level-3 regional boundaries from the Bundesamt für Kartographie und Geodäsie (BKG). The shapefile is extracted from a ZIP archive on first run.

Both layers are reprojected to **EPSG:3035** (ETRS89-LAEA Europe) — a metric equal-area CRS suited for area-based operations in Germany. This ensures that intersection areas computed in the next step are comparable across the country.

In [4]:
plz_gpkg = r'PLZ_Gebiete.gpkg'
zip_file = r'nuts250_12-31.gk3.shape.zip'
nuts3_shp = r'extracted_shapefile\nuts250_12-31.gk3.shape\nuts250_1231\NUTS250_N3.shp'

gdf_plz = gpd.read_file(plz_gpkg)

# read polygons
try:
    nuts3_gdf = gpd.read_file(nuts3_shp)
except Exception:
    print(f"Shapefile not found, extracting from {zip_file}...")
    extract_dir = "extracted_shapefile"
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(extract_dir)
    nuts3_gdf = gpd.read_file(nuts3_shp)

target_crs = 'EPSG:3035'
if gdf_plz.crs != target_crs:
    gdf_plz = gdf_plz.to_crs(target_crs)
if nuts3_gdf.crs != target_crs:
    nuts3_gdf = nuts3_gdf.to_crs(target_crs)

## 5. Spatial Join: PLZ → NUTS3

This section defines the core spatial linking logic and applies it to all PLZ precisions present in `dOI`.

### Functions

**`reduce_plz_precision(gdf, digits)`**  
Truncates the 5-digit PLZ to 2–5 characters and dissolves overlapping geometries into a single polygon per truncated code. This produces the coarser regional zones required when input data only contains truncated PLZ values.

**`assign_nuts3_by_area(gdf_plz_reduced, gdf_nuts3)`**  
Computes a geometric intersection overlay between PLZ zones and NUTS3 polygons, then assigns each PLZ zone to the NUTS3 region with the **largest intersection area** (best-fit by land area). Returns a lookup table with columns `plz`, `NUTS_CODE`, `NUTS_NAME`.

**`build_flexible_mapping(dOI, gdf_plz, gdf_nuts3)`**  
Inspects `dOI` for all distinct PLZ digit lengths and builds a separate mapping for each precision level. The results are concatenated into a single lookup table keyed on `(plz, _plz_len)`, which allows a mixed-precision dataset to be joined correctly in one step.

### Why area-based assignment?

PLZ boundaries do not align with NUTS3 boundaries. A postal code may straddle two or more NUTS3 regions. Assigning by largest intersection area is a standard, reproducible heuristic that minimises misclassification for most use cases, especially for 5-digit codes where the overlap is typically dominated by one region.

In [5]:
def reduce_plz_precision(gdf: gpd.GeoDataFrame, digits: int) -> gpd.GeoDataFrame:
    if digits not in (2, 3, 4, 5):
        raise ValueError("digits must be 2, 3, 4 or 5")
    result = gdf.copy()
    result["plz"] = result["plz"].astype(str).str.zfill(5).str[:digits]
    return result.dissolve(by="plz", as_index=False)


def assign_nuts3_by_area(gdf_plz_reduced, gdf_nuts3):
    overlay = gpd.overlay(
        gdf_plz_reduced[['plz', 'geometry']],
        gdf_nuts3[['NUTS_CODE', 'NUTS_NAME', 'geometry']],
        how='intersection'
    )
    overlay['intersection_area'] = overlay.geometry.area
    idx = overlay.groupby('plz')['intersection_area'].idxmax()
    return overlay.loc[idx, ['plz', 'NUTS_CODE', 'NUTS_NAME']].copy()


def build_flexible_mapping(dOI, gdf_plz, gdf_nuts3):
    plz_lengths = dOI['plz'].dropna().str.len().unique()
    mappings = []
    for length in sorted(plz_lengths):
        if length not in (2, 3, 4, 5):
            print(f"  Warning: skipping PLZ length {length} (not in 2-5)")
            continue
        print(f"  Building NUTS3 mapping for {length}-digit PLZ...")
        reduced = reduce_plz_precision(gdf_plz, digits=length)
        mapping = assign_nuts3_by_area(reduced, gdf_nuts3)
        mapping['_plz_len'] = length
        mappings.append(mapping)
    return pd.concat(mappings, ignore_index=True)


# --- link ---
dOI['_plz_len'] = dOI['plz'].str.len()
mapping = build_flexible_mapping(dOI, gdf_plz, nuts3_gdf)

dOI = dOI.merge(
    mapping,
    on=['plz', '_plz_len'],
    how='left'
).drop(columns=['_plz_len'])

print(f"Matched {dOI['NUTS_CODE'].notna().sum()} / {len(dOI)} rows to NUTS3")
dOI

  Building NUTS3 mapping for 2-digit PLZ...
  Building NUTS3 mapping for 3-digit PLZ...
  Building NUTS3 mapping for 4-digit PLZ...
  Building NUTS3 mapping for 5-digit PLZ...
Matched 29 / 29 rows to NUTS3


,id,plz,Encounter_period_start,Condition_recordedDate,Patient_gender,Patient_birthDate,Feststellungsdatum,NUTS_CODE,NUTS_NAME
0,0,1011,NaT,2024-08-15,male,1969-01-01,2019-05-14,DE300,Berlin
1,0,1011,2019-05-14,2019-05-14,male,1969-01-01,2019-05-14,DE300,Berlin
2,1,20095,2023-10-12,2023-10-12,male,1939-01-01,2019-05-14,DE600,Hamburg
3,2,80331,2022-11-08,2022-11-08,male,1994-01-01,2019-05-14,DE212,"München, Kreisfreie Stadt"
4,3,40210,2024-06-02,2024-06-02,other,1981-01-01,2019-05-14,DEA11,"Düsseldorf, Kreisfreie Stadt"
5,4,50667,2021-09-12,2021-09-12,female,1982-01-01,2019-05-14,DEA23,"Köln, Kreisfreie Stadt"
6,5,6031,2017-06-19,2017-06-19,male,1977-01-01,2019-05-01,DE712,"Frankfurt am Main, Kreisfreie Stadt"
7,6,70173,2022-10-05,2022-10-05,female,1958-01-01,2019-05-01,DE111,"Stuttgart, Stadtkreis"
8,7,30159,2024-02-20,2024-02-20,female,1981-01-01,NaT,DE929,Region Hannover
9,8,010,2023-04-09,2023-04-09,other,1970-01-01,NaT,DED21,"Dresden, Kreisfreie Stadt"


## 6. Age Calculation and Date Anonymisation

To protect patient privacy, exact dates are replaced with **ISO calendar-week strings** (`YYYY_WW`), and patient birth dates are removed after deriving ages.

### Age computation

`age_in_years(event_series, birth_series)` computes the number of complete years between the birth date and a given event date using:
```
age = floor( (event_date − birth_date).days / 365.25 )
```
Using 365.25 accounts for leap years. The result is a nullable integer (`Int64`); rows where either date is missing return `pd.NA`.

Ages are computed **before** overwriting the date columns, because `to_year_week` would destroy the datetime type needed for subtraction.

### Calendar-week encoding

`to_year_week(dt_series)` uses the ISO 8601 week numbering system (Monday = start of week) and formats output as `YYYY_WW` (zero-padded week number). Missing dates become empty strings.

### Output columns

| Column | Type | Description |
|---|---|---|
| `id` | int | Surrogate patient identifier |
| `Patient_gender` | str | Gender as recorded in FHIR |
| `Encounter_period_start` | str | Encounter start, as `YYYY_WW` |
| `Condition_recordedDate` | str | Condition documentation date, as `YYYY_WW` |
| `Feststellungsdatum` | str | Diagnosis confirmation date, as `YYYY_WW` |
| `NUTS_CODE` | str | NUTS3 region code |
| `NUTS_NAME` | str | NUTS3 region name |
| `age_encounter_start` | Int64 | Patient age at encounter start |
| `age_recordedDate` | Int64 | Patient age at condition documentation |
| `age_Feststellungsdatum` | Int64 | Patient age at diagnosis confirmation |

In [6]:
def to_year_week(dt_series):
    """Parsed datetime -> 'YYYY_WW' string. NaT -> empty string."""
    iso = dt_series.dt.isocalendar()
    result = (
        iso["year"].astype("Int64").astype(str) + "_" +
        iso["week"].astype("Int64").astype(str).str.zfill(2)
    )
    result[dt_series.isna()] = ""
    return result

def age_in_years(event_series, birth_series):
    """Full years between birthdate and event date. NaT -> pd.NA."""
    days = (event_series - birth_series).dt.days
    age = np.floor(days / 365.25).astype("Int64")
    age[event_series.isna() | birth_series.isna()] = pd.NA
    return age

# Compute ages first (before overwriting the date columns)
dOI["age_encounter_start"]    = age_in_years(dOI["Encounter_period_start"], dOI["Patient_birthDate"])
dOI["age_recordedDate"]       = age_in_years(dOI["Condition_recordedDate"],  dOI["Patient_birthDate"])
dOI["age_Feststellungsdatum"] = age_in_years(dOI["Feststellungsdatum"],      dOI["Patient_birthDate"])

# Convert date columns to year-week
dOI["Encounter_period_start"] = to_year_week(dOI["Encounter_period_start"])
dOI["Condition_recordedDate"] = to_year_week(dOI["Condition_recordedDate"])
dOI["Feststellungsdatum"]     = to_year_week(dOI["Feststellungsdatum"])

# Drop the now-redundant birthdate
dOI = dOI.drop(columns=["Patient_birthDate"])
dOI = dOI.drop(columns=["plz"])

dOI

,id,Encounter_period_start,Condition_recordedDate,Patient_gender,Feststellungsdatum,NUTS_CODE,NUTS_NAME,age_encounter_start,age_recordedDate,age_Feststellungsdatum
0,0,,2024_33,male,2019_20,DE300,Berlin,<NA>,55,50
1,0,2019_20,2019_20,male,2019_20,DE300,Berlin,50,50,50
2,1,2023_41,2023_41,male,2019_20,DE600,Hamburg,84,84,80
3,2,2022_45,2022_45,male,2019_20,DE212,"München, Kreisfreie Stadt",28,28,25
4,3,2024_22,2024_22,other,2019_20,DEA11,"Düsseldorf, Kreisfreie Stadt",43,43,38
5,4,2021_36,2021_36,female,2019_20,DEA23,"Köln, Kreisfreie Stadt",39,39,37
6,5,2017_25,2017_25,male,2019_18,DE712,"Frankfurt am Main, Kreisfreie Stadt",40,40,42
7,6,2022_40,2022_40,female,2019_18,DE111,"Stuttgart, Stadtkreis",64,64,61
8,7,2024_08,2024_08,female,,DE929,Region Hannover,43,43,<NA>
9,8,2023_14,2023_14,other,,DED21,"Dresden, Kreisfreie Stadt",53,53,<NA>


## 7. Save Output

The final DataFrame is written to `result.csv` (no index column). This file contains all columns described in the output table above and can be used directly for epidemiological analyses or passed to downstream visualisation tools.

In the production script `link_plz_nuts3.py`, the output path defaults to `result.csv` in the working directory but can be overridden via the `-o` / `--output` argument.

In [7]:
print('Saving result...')
dOI.to_csv('result.csv', index=False)
print(f'Done. {len(dOI)} rows written to result.csv')

Saving result...
Done. 29 rows written to result.csv
